In [2]:
print("hello World")

hello World


In [2]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
df = pd.read_csv('complaints.csv')
print(df.head())

  Date received                                            Product  \
0    2020-07-06  Credit reporting, credit repair services, or o...   
1    2019-12-26                        Credit card or prepaid card   
2    2020-05-08  Credit reporting, credit repair services, or o...   
3    2024-01-05  Credit reporting or other personal consumer re...   
4    2024-01-21  Credit reporting or other personal consumer re...   

                                  Sub-product  \
0                            Credit reporting   
1  General-purpose credit card or charge card   
2                            Credit reporting   
3                            Credit reporting   
4                            Credit reporting   

                                               Issue  \
0               Incorrect information on your report   
1  Advertising and marketing, including promotion...   
2               Incorrect information on your report   
3               Incorrect information on your report   
4   

In [4]:
print(df.columns)

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='object')


In [5]:
print(df["Product"].unique())

['Credit reporting, credit repair services, or other personal consumer reports'
 'Credit card or prepaid card'
 'Credit reporting or other personal consumer reports' 'Student loan'
 'Checking or savings account' 'Debt collection' 'Credit card' 'Mortgage'
 'Money transfer, virtual currency, or money service'
 'Payday loan, title loan, or personal loan' 'Consumer Loan'
 'Vehicle loan or lease' 'Bank account or service' 'Money transfers'
 'Prepaid card' 'Payday loan, title loan, personal loan, or advance loan'
 'Credit reporting' 'Debt or credit management' 'Payday loan'
 'Other financial service' 'Virtual currency']


In [6]:
df.shape[0]

13777727

In [7]:
df1=df[df["Product"].isin([
    "Credit card",
    "Credit card or prepaid card",
])]

df1.shape[0]

497978

Your mapping:

Company response	Consumer disputed?	dispute	not_dispute	escalate
Closed with monetary relief	No	✔		
Closed with explanation	No		✔	
Closed with explanation	Yes	✔		
Closed without relief	NaN			✔


1️⃣ Closed with monetary relief + Consumer disputed = No → dispute

Reason

“Closed with monetary relief” means the company gave money back or compensated the consumer.

Examples:

refund issued

charge reversed

fee waived

So the company effectively acknowledged the complaint had merit.

Even though:

Consumer disputed? = No

that only means the consumer accepted the resolution, not that the complaint was invalid.

Example:

Complaint:

"My credit card was charged twice."

Company action:

Refund issued.

Consumer reaction:

Accepted response.

So it is clearly a valid dispute.

2️⃣ Closed with explanation + Consumer disputed = No → not_dispute

Reason

“Closed with explanation” means the company explained the situation and found no error.

Examples:

charge was valid

interest calculation correct

policy explained

Consumer also accepted the explanation.

So the complaint likely resulted from misunderstanding rather than an actual issue.

Example:

Complaint:

"I was charged interest unexpectedly."

Company response:

Interest policy explained.

Consumer:

Accepted explanation.

So:

not_dispute


3️⃣ Closed with explanation + Consumer disputed = Yes → dispute

Reason

Here the company says no problem, but the consumer still disagrees.

Meaning:

issue not resolved

consumer still believes there is an error

Example:

Complaint:

"My credit card interest is wrong."

Company response:

Interest calculation explained.

Consumer:

Still disputes.

This indicates an unresolved conflict, so your system should treat it as a dispute.

4️⃣ Closed without relief + Consumer disputed = NaN → escalate_to_human

Reason

“Closed without relief” means:

company did not provide compensation

unclear if complaint was valid

Consumer disputed? = NaN means:

no follow-up response from consumer

missing information

So the outcome is uncertain.

Example:

Complaint:

Unauthorized charge reported.

Company response:

Closed without relief.

Consumer reaction:

Unknown.

The system cannot confidently decide.

So safest option:

escalate_to_human
Final Logic Summary

Your rules are essentially:

1. Monetary relief → dispute
2. Explanation + consumer agrees → not_dispute
3. Explanation + consumer disagrees → dispute
4. Unclear outcome → escalate

This is a reasonable labeling strategy for training / RAG decision logic.

In [8]:
def map_label(response, disputed):

    # normalize values
    response = str(response).strip().lower()
    disputed = str(disputed).strip().lower()

    # Rule 1: monetary relief → dispute
    if response == "closed with monetary relief":
        return "dispute"

    # Rule 2: explanation + consumer accepts → not_dispute
    elif response == "closed with explanation" and disputed == "no":
        return "not_dispute"

    # Rule 3: explanation + consumer disagrees → dispute
    elif response == "closed with explanation" and disputed == "yes":
        return "dispute"

    # Rule 4: closed without relief + missing dispute info → escalate
    elif response == "closed without relief" and disputed in ["nan", "none"]:
        return "escalate_to_human"

    # fallback
    else:
        return "escalate_to_human"

In [9]:
df1.loc[:, "label"] = df1.apply(
    lambda row: map_label(
        row["Company response to consumer"],
        row["Consumer disputed?"]
    ),
    axis=1
)

C:\Users\harpi\AppData\Local\Temp\ipykernel_13060\3671361293.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.loc[:, "label"] = df1.apply(


In [10]:
print(df1.columns)

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID', 'label'],
      dtype='object')


In [11]:
print(df1["label"].unique())

['escalate_to_human' 'dispute' 'not_dispute']


In [12]:
print(df1["label"].head())

1     escalate_to_human
15    escalate_to_human
19              dispute
21    escalate_to_human
32              dispute
Name: label, dtype: object


In [13]:
df1.drop(columns=["Company response to consumer"], inplace=True)
df1.drop(columns=["Consumer disputed?"], inplace=True)

C:\Users\harpi\AppData\Local\Temp\ipykernel_13060\1349815727.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop(columns=["Company response to consumer"], inplace=True)
C:\Users\harpi\AppData\Local\Temp\ipykernel_13060\1349815727.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop(columns=["Consumer disputed?"], inplace=True)


In [14]:
print(df1.columns)

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Timely response?',
       'Complaint ID', 'label'],
      dtype='object')


In [15]:
df1["text"] = (
    "Product: " + df1["Product"].astype(str) +
    " Sub-product: " + df1["Sub-product"].astype(str) +
    " Issue: " + df1["Issue"].astype(str) +
    " Sub-issue: " + df1["Sub-issue"].astype(str) +
    " Complaint: " + df1["Consumer complaint narrative"].astype(str)
)

C:\Users\harpi\AppData\Local\Temp\ipykernel_13060\841606324.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["text"] = (


In [16]:
df1[["text", "label"]].head()

,text,label
1,Product: Credit card or prepaid card Sub-produ...,escalate_to_human
15,Product: Credit card or prepaid card Sub-produ...,escalate_to_human
19,Product: Credit card Sub-product: General-purp...,dispute
21,Product: Credit card or prepaid card Sub-produ...,escalate_to_human
32,Product: Credit card Sub-product: nan Issue: L...,dispute


In [17]:
df1 = df1[df1["Consumer complaint narrative"].notna()]

In [18]:
df1 = df1[df1["Consumer complaint narrative"].str.len() > 20]

In [19]:
df1.shape[0]

218674

In [20]:
df1.to_csv("processed_complaints.csv", index=False)